# Agentic AI: Tool Calling & LangGraph Orchestration

This notebook explores the implementation of **Agentic Workflows** using LangChain and LangGraph. It transitions from understanding the manual **"Reason-Act" (ReAct)** loop to using high-level abstractions for automated decision-making.

### Key Learning Objectives:
* **Manual Tool Loops:** Deep dive into the handshaking process between LLMs and Python functions (Tool Calling).
* **Custom Tool Definition:** Using the `@tool` decorator with precise type-hinting and docstrings.
* **LangGraph Integration:** Implementing the modern `create_react_agent` for scalable, stateful agentic behavior.
* **Real-world Utilities:** Practical examples including a budget calculator, weather service, and a safe calculator tool.

In [1]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

import os
from langchain_groq import ChatGroq

from dotenv import load_dotenv
load_dotenv()

# Initialize the Groq model cleanly
llama_llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=256
)

In [2]:
llama_llm.invoke("Hello, world!")

AIMessage(content="Hello, world. It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 39, 'total_tokens': 66, 'completion_time': 0.041417462, 'completion_tokens_details': None, 'prompt_time': 0.001805654, 'prompt_tokens_details': None, 'queue_time': 0.005574831, 'total_time': 0.043223116}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0562-d6fb-7b92-9707-c0fdd9c190b6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 27, 'total_tokens': 66})

## Tools

In [3]:
from langchain_core.tools import tool

In [4]:
# ---------------------------------------------------------
# Define Custom Tools (The @tool decorator)
# ---------------------------------------------------------
# MANDATORY: We must use type hints (location: str) and a clear docstring.
# The LLM reads the docstring to figure out when to trigger this function.

@tool
def get_current_weather(location: str) -> str:
    """Fetch the current weather for a specific city or location."""
    # In a real app, we would hit the OpenWeather API here.
    if "calgary" in location.lower():
        return "It is 15°C and partly cloudy in Calgary."
    return f"It is 22°C and sunny in {location}."

@tool
def calculate_internship_budget(flights: float, monthly_rent: float) -> str:
    """Calculate the total estimated budget for a 3-month summer research internship."""
    total = flights + (monthly_rent * 3)
    return f"Total estimated budget needed: ${total}"

In [5]:
# ---------------------------------------------------------
# Assemble the "Toolkit"
# ---------------------------------------------------------
# In modern LCEL, a toolkit is simply a standard Python list of your tools.
my_tools = [get_current_weather, calculate_internship_budget]

In [6]:
# ---------------------------------------------------------
# Tool Binding
# ---------------------------------------------------------
# We physically fuse the tools into the Llama 3 model. 
# The model now knows these tools exist and can use them.
llm_with_tools = llama_llm.bind_tools(my_tools)

In [18]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

print("=== STARTING AGENT LOOP ===")

# We must explicitly tell the LLM that its job is to summarize tool outputs.
messages = [
    SystemMessage(content="You are a helpful AI assistant. You MUST synthesize and summarize the results of any tools you use into a friendly conversational response for the user."),
    HumanMessage(content="What should I wear today if I am in Calgary?")
]

print("\n1. Asking LLM to evaluate the question...")
ai_response = llm_with_tools.invoke(messages)
messages.append(ai_response) # Save the LLM's request to memory

tool_call = ai_response.tool_calls[0]
print(f"-> LLM requested tool: '{tool_call['name']}' with args: {tool_call['args']}")

print("\n2. Executing Python Tool...")
# LangChain executes the function safely using the LLM's arguments
python_result = get_current_weather.invoke(tool_call['args'])
print(f"-> Python retrieved: '{python_result}'")

print("\n3. Handing data back to the LLM...")
tool_message = ToolMessage(
    content=str(python_result),
    tool_call_id=tool_call["id"] 
)
messages.append(tool_message)

print("\n4. Generating Final Answer...")
final_response = llm_with_tools.invoke(messages)

# ---------------------------------------------------------
# THE FAIL-SAFE INSPECTION
# ---------------------------------------------------------
print("\nFINAL AI RESPONSE:")
print("-" * 50)

# If the LLM generated text, print it.
if final_response.content:
    print(final_response.content)
# If it is STILL empty, it means the LLM is trying to call another tool!
else:
    print("[WARNING] The LLM refused to write text. It requested another tool instead:")
    print(final_response.tool_calls)

=== STARTING AGENT LOOP ===

1. Asking LLM to evaluate the question...
-> LLM requested tool: 'get_current_weather' with args: {'location': 'Calgary'}

2. Executing Python Tool...
-> Python retrieved: 'It is 15°C and partly cloudy in Calgary.'

3. Handing data back to the LLM...

4. Generating Final Answer...

FINAL AI RESPONSE:
--------------------------------------------------
You should dress in layers for today, with a light jacket or sweater to keep you warm. It's a good idea to bring an umbrella or raincoat, as there's a chance of rain.


In [10]:
from langchain_core.messages import HumanMessage, ToolMessage

print("=== TEST 2: The Complete Agent Execution Loop ===")

# 1. Start the Conversation Memory
messages = [
    HumanMessage(content="My flights are $1200 and rent is $700/month. What's my 3-month budget?")
]

# ---------------------------------------------------------
# PHASE 1: The Request
# ---------------------------------------------------------
print("1. Asking LLM...")
ai_response = llm_with_tools.invoke(messages)

# We must append the AI's tool request to the memory so it remembers what it asked for
messages.append(ai_response) 

# Extract the tool call
tool_call = ai_response.tool_calls[0]
print(f"-> LLM Halted text generation. Requested Tool: {tool_call['name']}")
print(f"-> With Arguments: {tool_call['args']}")

# ---------------------------------------------------------
# PHASE 2: The Execution (Running the Python Code)
# ---------------------------------------------------------
print("\n2. Executing Python Function...")
# We use the tool's .invoke() method to safely pass the LLM's arguments into our Python function
python_result = calculate_internship_budget.invoke(tool_call["args"])
print(f"-> Python calculated: {python_result}")

# ---------------------------------------------------------
# PHASE 3: The Handoff 
# ---------------------------------------------------------
print("\n3. Handing result back to LLM...")
# We create a special "ToolMessage" to prove to the LLM that we ran its requested tool
tool_message = ToolMessage(
    content=str(python_result),
    tool_call_id=tool_call["id"] # MANDATORY: Links this answer to the LLM's specific request
)
messages.append(tool_message)

# ---------------------------------------------------------
# PHASE 4: The Final Answer
# ---------------------------------------------------------
# We invoke the LLM one last time with the complete history.
# It reads the ToolMessage, realizes the math is done, and finally generates text.
final_response = llm_with_tools.invoke(messages)

print("\nFINAL AI RESPONSE:")
print("-" * 50)
print(final_response.content)

=== TEST 2: The Complete Agent Execution Loop ===
1. Asking LLM...
-> LLM Halted text generation. Requested Tool: calculate_internship_budget
-> With Arguments: {'flights': 1200, 'monthly_rent': 700}

2. Executing Python Function...
-> Python calculated: Total estimated budget needed: $3300.0

3. Handing result back to LLM...

FINAL AI RESPONSE:
--------------------------------------------------
The total estimated budget for a 3-month summer research internship is $3300.


In [11]:
messages

[HumanMessage(content="My flights are $1200 and rent is $700/month. What's my 3-month budget?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fb8c9wwh7', 'function': {'arguments': '{"flights":1200,"monthly_rent":700}', 'name': 'calculate_internship_budget'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 326, 'total_tokens': 352, 'completion_time': 0.03174365, 'completion_tokens_details': None, 'prompt_time': 0.024202174, 'prompt_tokens_details': None, 'queue_time': 0.005363906, 'total_time': 0.055945824}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0529-bc51-7222-93a3-663cedbcd291-0', tool_calls=[{'name': 'calculate_internship_budget', 'args': {'flights': 1200, 'monthly_rent': 700}, 'id': 'fb8c9wwh7', 'type': 'tool_cal

## Agents

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

In [13]:
@tool
def get_current_weather(location: str) -> str:
    """Fetch the current weather for a specific city or location."""
    if "calgary" in location.lower():
        return "It is 15°C and partly cloudy in Calgary."
    return f"It is 22°C and sunny in {location}."

@tool
def calculate_internship_budget(flights: float, monthly_rent: float) -> str:
    """Calculate the total estimated budget for a 3-month summer research internship."""
    # Ensure inputs are cast to floats in case the LLM passes them as strings
    total = float(flights) + (float(monthly_rent) * 3)
    return f"Total estimated budget needed: ${total}"

In [14]:
my_tools = [get_current_weather, calculate_internship_budget]

In [15]:
agent_executor = create_react_agent(
    model=llama_llm, 
    tools=my_tools
)

In [16]:
print("=== EXECUTING MODERN LANGGRAPH AGENT ===\n")

query = "My flights to Calgary are 1200 and rent is 700. What is my total budget, and what is the weather there right now?"

# LangGraph state requires a dictionary with a 'messages' array, not an 'input' string
inputs = {"messages": [HumanMessage(content=query)]}

# Execute the graph
result = agent_executor.invoke(inputs)

print("\n" + "="*50)
print("FINAL ANSWER EXTRACTED:")
print("="*50)

# The result is a dictionary containing the entire conversation history.
# The final response from the LLM is always the last message in the array [-1].
print(result["messages"][-1].content)

=== EXECUTING MODERN LANGGRAPH AGENT ===


FINAL ANSWER EXTRACTED:
Total estimated budget needed: $3300.0
It is 15°C and partly cloudy in Calgary.


In [17]:
result

{'messages': [HumanMessage(content='My flights to Calgary are 1200 and rent is 700. What is my total budget, and what is the weather there right now?', additional_kwargs={}, response_metadata={}, id='27da8f0e-eaa2-467d-a816-e35e0b02f5d6'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'd2yz84wrk', 'function': {'arguments': '{"flights":1200,"monthly_rent":700}', 'name': 'calculate_internship_budget'}, 'type': 'function'}, {'id': 'sy7xqp9g6', 'function': {'arguments': '{"flights":1200,"monthly_rent":700}', 'name': 'calculate_internship_budget'}, 'type': 'function'}, {'id': '6gyrpkwbm', 'function': {'arguments': '{"location":"Calgary"}', 'name': 'get_current_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 334, 'total_tokens': 400, 'completion_time': 0.086513627, 'completion_tokens_details': None, 'prompt_time': 0.024766194, 'prompt_tokens_details': None, 'queue_time': 0.00632864, 'total_time': 0.111279821}, 

In [21]:
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

In [22]:
@tool
def calculator(a: float, b: float, operator: str) -> str:
    """A safe calculator tool.
    'a' and 'b' are the numbers. 'operator' must be strictly one of: 'add', 'subtract', 'multiply', 'divide'.
    """
    try:
        if operator == "add":
            return f"Result: {a + b}"
        elif operator == "subtract":
            return f"Result: {a - b}"
        elif operator == "multiply":
            return f"Result: {a * b}"
        elif operator == "divide":
            if b == 0:
                return "Error: Cannot divide by zero."
            return f"Result: {a / b}"
        else:
            return f"Error: Unknown operator '{operator}'."
    except Exception as e:
        return f"Error calculating: {str(e)}"

@tool
def format_text(text: str, format_type: str) -> str:
    """Format a given string.
    'text' is the raw string. 'format_type' must be strictly one of: 'uppercase', 'lowercase', 'titlecase'.
    """
    try:
        if format_type.lower() == "uppercase":
            return text.upper()
        elif format_type.lower() == "lowercase":
            return text.lower()
        elif format_type.lower() == "titlecase":
            return text.title()
        else:
            return f"Error: Unknown format type '{format_type}'."
    except Exception as e:
        return f"Error formatting text: {str(e)}"

# Assemble the toolkit array
my_tools = [calculator, format_text]

# ---------------------------------------------------------
# 3. Compile the LangGraph Agent
# ---------------------------------------------------------
agent_executor = create_react_agent(
    model=llama_llm, 
    tools=my_tools
)

In [23]:
# ---------------------------------------------------------
# 4. Execution & Testing
# ---------------------------------------------------------
test_questions = [
    "What is 25 + 63?", 
    "Can you convert 'hello world' to uppercase?"
]

print("=== EXECUTING MODERN LANGGRAPH AGENT ===\n")

system_msg = SystemMessage(content="""You are a helpful assistant. 
Once a tool provides a result, summarize that result for the user and STOP. 
Do not call the same tool repeatedly with the same input.""")

for question in test_questions:
    print(f"\n===== Testing: {question} =====")
    
    # LangGraph requires a dictionary with a 'messages' array
    inputs = {"messages": [system_msg, HumanMessage(content=question)]}
    
    # Execute the graph
    result = agent_executor.invoke(inputs)
    
    # Extract the final answer from the last message in the array
    final_answer = result["messages"][-1].content
    print(f"Final Answer: {final_answer}")

=== EXECUTING MODERN LANGGRAPH AGENT ===


===== Testing: What is 25 + 63? =====
Final Answer: The result of the calculation is 88.0.

===== Testing: Can you convert 'hello world' to uppercase? =====
Final Answer: The function 'format_text' has been used to convert 'hello world' to uppercase. The result is 'HELLO WORLD'.


In [25]:
result

{'messages': [SystemMessage(content='You are a helpful assistant. \nOnce a tool provides a result, summarize that result for the user and STOP. \nDo not call the same tool repeatedly with the same input.', additional_kwargs={}, response_metadata={}, id='34e7246a-ff6a-442a-adce-ab7901653812'),
  HumanMessage(content="Can you convert 'hello world' to uppercase?", additional_kwargs={}, response_metadata={}, id='f3d9ef3d-1be3-4328-8261-fc65610747a9'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kk0ngce9z', 'function': {'arguments': '{"format_type":"uppercase","text":"hello world"}', 'name': 'format_text'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 457, 'total_tokens': 479, 'completion_time': 0.053293886, 'completion_tokens_details': None, 'prompt_time': 0.027581524, 'prompt_tokens_details': None, 'queue_time': 0.005136652, 'total_time': 0.08087541}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'f